<a href="https://colab.research.google.com/github/Hackathon-D-Group-2026/Hackathon_D_Group_2026/blob/main/seasonal_flu_vaccine_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Seasonal Flu Vaccine Acceptance & Cost Optimization Pipeline

This Jupyter Notebook implements an end-to-end Machine Learning and Cost-Optimization pipeline based on **Dataset A** and the **Target Metric Information**.

### Objective
Minimize the total financial cost incurred by the town, which consists of:
1. **Vaccine Costs:** $300 per vaccine sent/administered (vaccines sent but unused are ruined and still incur cost).
2. **Hospitalization Costs:** $18,500 per hospitalization for H3N2 flu.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Data Exploration (EDA)

We load and inspect Dataset A (or simulate its exact structure if running stand-alone).
Features include behavioral attributes, doctor recommendations, opinions on vaccine effectiveness/risk, demographic data, and chronic medical condition counts.


In [2]:
# Simulated generation of Dataset A structure (Replace this section with pd.read_csv('dataset_a.csv'))
n_samples = 5000

data = {
    'respondent_id': range(n_samples),
    'flu_concern': np.random.choice([0, 1, 2, 3], size=n_samples, p=[0.1, 0.4, 0.3, 0.2]),
    'flu_knowledge': np.random.choice([0, 1, 2], size=n_samples, p=[0.2, 0.5, 0.3]),
    'behavioral_antiviral_meds': np.random.choice([0, 1], size=n_samples, p=[0.9, 0.1]),
    'behavioral_avoidance': np.random.choice([0, 1], size=n_samples, p=[0.3, 0.7]),
    'behavioral_face_mask': np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2]),
    'behavioral_wash_hands': np.random.choice([0, 1], size=n_samples, p=[0.1, 0.9]),
    'doctor_recc_seasonal': np.random.choice([0, 1], size=n_samples, p=[0.7, 0.3]),
    'health_worker': np.random.choice([0, 1], size=n_samples, p=[0.85, 0.15]),
    'opinion_seas_vacc_effective': np.random.choice([1, 2, 3, 4, 5], size=n_samples, p=[0.05, 0.1, 0.15, 0.3, 0.4]),
    'opinion_seas_risk': np.random.choice([1, 2, 3, 4, 5], size=n_samples, p=[0.2, 0.3, 0.2, 0.2, 0.1]),
    'age_group': np.random.choice(['18-34', '35-54', '55-64', '65+'], size=n_samples),
    'education': np.random.choice(['< 12 Years', '12 Years', 'College Graduate', 'Some College'], size=n_samples),
    'income_poverty': np.random.choice(['Below Poverty', '<= $75,000', '> $75,000'], size=n_samples),
    'number_chronic_med_conditions': np.random.choice([0, 1, 2, 3], size=n_samples, p=[0.5, 0.3, 0.15, 0.05]),
    'seasonal_vaccine': np.random.choice([0, 1], size=n_samples, p=[0.53, 0.47])
}

df = pd.DataFrame(data)

print(f"Dataset Shape: {df.shape}")
print("\nMissing values per column:")
print(df.isnull().sum())

print("\nTarget Variable Distribution (`seasonal_vaccine`):")
print(df['seasonal_vaccine'].value_counts(normalize=True))


Dataset Shape: (5000, 16)

Missing values per column:
respondent_id                    0
flu_concern                      0
flu_knowledge                    0
behavioral_antiviral_meds        0
behavioral_avoidance             0
behavioral_face_mask             0
behavioral_wash_hands            0
doctor_recc_seasonal             0
health_worker                    0
opinion_seas_vacc_effective      0
opinion_seas_risk                0
age_group                        0
education                        0
income_poverty                   0
number_chronic_med_conditions    0
seasonal_vaccine                 0
dtype: int64

Target Variable Distribution (`seasonal_vaccine`):
seasonal_vaccine
0    0.5278
1    0.4722
Name: proportion, dtype: float64


## 2. Preprocessing

We encode categorical features, drop non-informative identifiers (`respondent_id`), and perform a stratified train-test split.


In [3]:
# Separate features and target
X = df.drop(columns=['respondent_id', 'seasonal_vaccine'])
y = df['seasonal_vaccine']

# Encode categorical variables using One-Hot Encoding
X = pd.get_dummies(X, drop_first=True)

# Train-Test Split (80% train, 20% validation/test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")


Training set shape: (4000, 19)
Testing set shape: (1000, 19)


## 3. Modelling

- **Vaccine Propensity Model:** A `RandomForestClassifier` trained to predict the probability that a person will accept the seasonal vaccine.
- **Hospitalization Risk Function:** Models individual baseline hospitalization risk based on chronic conditions/health status and assesses risk reduction when vaccinated.


In [4]:
# Model 1: Predict Vaccine Acceptance
vaccine_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    random_state=RANDOM_STATE
)
vaccine_model.fit(X_train, y_train)

# Predict probabilities for test set
y_pred_prob = vaccine_model.predict_proba(X_test)[:, 1]

# Model 2: Estimate Hospitalization Probability
def estimate_hospitalization_risk(features_df, vaccination_status):
    # Base risk derived from chronic conditions
    base_risk = 0.05 + (features_df.get('number_chronic_med_conditions', 0) * 0.12)
    # Vaccine reduces risk by 80%
    risk = np.where(vaccination_status == 1, base_risk * 0.2, base_risk)
    return np.clip(risk, 0.01, 0.95)


## 4. Performance Analysis & Cost Optimization

We simulate decision thresholds to select individuals who should be sent a vaccine in order to minimize the total town financial liability:

$$\text{Total Cost} = (\text{Number of Vaccines Sent} \times \$300) + (\sum \text{Probability of Hospitalization} \times \$18,500)$$


In [5]:
COST_VACCINE = 300       # $300 per vaccine sent/administered
COST_HOSPITAL = 18500    # $18,500 per hospitalization

def evaluate_strategy(y_true_accept, prob_accept, features, threshold=0.5):
    pred_send_vaccine = (prob_accept >= threshold).astype(int)

    vaccine_costs = 0
    hospital_costs = 0

    for i in range(len(y_true_accept)):
        sent = pred_send_vaccine[i]

        if sent == 1:
            vaccine_costs += COST_VACCINE
            hosp_prob = estimate_hospitalization_risk(features.iloc[i], vaccination_status=1)
        else:
            vaccine_costs += 0
            hosp_prob = estimate_hospitalization_risk(features.iloc[i], vaccination_status=0)

        hospital_costs += hosp_prob * COST_HOSPITAL

    total_cost = vaccine_costs + hospital_costs
    return total_cost, vaccine_costs, hospital_costs

# Search across thresholds to find the minimum cost operating point
thresholds = np.linspace(0.1, 0.9, 17)
best_threshold = 0.5
min_total_cost = float('inf')
results_log = []

for th in thresholds:
    t_cost, v_cost, h_cost = evaluate_strategy(y_test, y_pred_prob, X_test, threshold=th)
    results_log.append({'Threshold': round(th, 2), 'Total Cost': t_cost, 'Vaccine Cost': v_cost, 'Hosp Cost': h_cost})
    if t_cost < min_total_cost:
        min_total_cost = t_cost
        best_threshold = th

results_df = pd.DataFrame(results_log)
print("Cost Evaluation Across Decision Thresholds:")
print(results_df)

print(f"\nOptimal Decision Threshold: {best_threshold:.2f}")
print(f"Minimum Total Cost Achieved: ${min_total_cost:,.2f}")


Cost Evaluation Across Decision Thresholds:
    Threshold  Total Cost  Vaccine Cost  Hosp Cost
0        0.10    837536.0        300000   537536.0
1        0.15    837536.0        300000   537536.0
2        0.20    837536.0        300000   537536.0
3        0.25    837536.0        300000   537536.0
4        0.30    853944.0        298500   555444.0
5        0.35    896048.0        292800   603248.0
6        0.40   1091040.0        265200   825840.0
7        0.45   1570400.0        186600  1383800.0
8        0.50   2152528.0         90000  2062528.0
9        0.55   2466584.0         34500  2432084.0
10       0.60   2638944.0          7800  2631144.0
11       0.65   2674824.0          1500  2673324.0
12       0.70   2687240.0           300  2686940.0
13       0.75   2687680.0             0  2687680.0
14       0.80   2687680.0             0  2687680.0
15       0.85   2687680.0             0  2687680.0
16       0.90   2687680.0             0  2687680.0

Optimal Decision Threshold: 0.10
Mini

## 5. Results & Feature Importance

Visualizing feature importances and classification metrics at the optimal decision threshold.


In [6]:
# Classification Metrics
y_pred = (y_pred_prob >= best_threshold).astype(int)
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_prob):.4f}")

# Feature Importances
importances = vaccine_model.feature_importances_
feature_imp_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': importances}).sort_values(by='Importance', ascending=False)
print("\nTop 10 Feature Importances:")
print(feature_imp_df.head(10))


--- Classification Report ---
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       528
           1       0.47      1.00      0.64       472

    accuracy                           0.47      1000
   macro avg       0.24      0.50      0.32      1000
weighted avg       0.22      0.47      0.30      1000

ROC-AUC Score: 0.5057

Top 10 Feature Importances:
                          Feature  Importance
9               opinion_seas_risk    0.111509
8     opinion_seas_vacc_effective    0.110413
0                     flu_concern    0.099651
10  number_chronic_med_conditions    0.091577
1                   flu_knowledge    0.077975
3            behavioral_avoidance    0.046661
6            doctor_recc_seasonal    0.044853
4            behavioral_face_mask    0.043497
18   income_poverty_Below Poverty    0.040171
7                   health_worker    0.038028


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
